In [ ]:
%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.signal import freqz
from ipywidgets import HTML
from IPython.display import display

# ============================================================
# COMPLETE OPTIMIZATION-BASED IIR LOW-PASS DESIGN
#
# No coefficients from a known solution are used.
#
# Starting data:
#   fp, fs, Fs, Ap, As, N
#
# The notebook calculates:
#   - random initial parameter vectors
#   - frequency grid
#   - weighting factors
#   - least-p optimization sequence
#   - final parameter vector xi
#   - second-order sections
#   - poles
#   - stability correction
#   - normalization correction
#   - final symbolic transfer function
#   - final specifications
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.io-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.io-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.io-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.io-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.io-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
    margin-bottom:5px;
}

.io-note{
    background:#fff9e8;
    border:1px solid #d9c477;
}

.io-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.io-col{
    flex:1;
    min-width:0;
}

.io-code{
    font-family:Consolas,monospace;
    font-size:12.5px;
}

.io-small{
    font-size:13px;
}

.io-table{
    border-collapse:collapse;
    font-size:13px;
    width:100%;
}

.io-table th,
.io-table td{
    text-align:center;
    padding:3px 8px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="io-root">

<div class="io-header">
Complete Optimization-Based IIR Low-Pass Design
</div>

<div class="io-doc">

This notebook performs the complete optimization procedure starting only from
the filter specifications. <b>No coefficients from a previously known solution
are used.</b>

<div style="text-align:center;font-size:15px;margin:6px 0;">
<b>
f<sub>p</sub> = 500 Hz,
&nbsp;
f<sub>s</sub> = 800 Hz,
&nbsp;
F<sub>s</sub> = 2000 Hz,
&nbsp;
A<sub>p</sub> = 0.25 dB,
&nbsp;
A<sub>s</sub> = 45 dB,
&nbsp;
N = 8.
</b>
</div>

For N = 8, the filter is represented as four second-order sections and the
optimization vector contains 17 unknown parameters:

<div style="text-align:center;font-size:14px;margin:6px 0;">
<b>
ξ = [α₀₁, α₁₁, β₀₁, β₁₁, ..., α₀₄, α₁₄, β₀₄, β₁₄, H₀].
</b>
</div>

The initial vector x₀ is generated randomly. A dense frequency grid containing
20 times as many samples as unknown parameters is constructed over the
passband and stopband. The transition band is excluded from the optimization.

The weighted approximation error is

<div style="text-align:center;font-size:14px;margin:6px 0;">
<b>
eᵢ(x) = wᵢ[M(x,ωᵢ) - M₀(ωᵢ)],
</b>
</div>

with M₀ = 1 in the passband and M₀ = 0 in the stopband.

The minimax solution is approached by a sequence of least-p problems. The
process starts with p = 2 and doubles p after each outer iteration. Each
least-p minimization is performed by SciPy's BFGS quasi-Newton optimizer.

The process terminates when the change in the maximum weighted error becomes
smaller than ε₁. The number of outer iterations is therefore <b>calculated by
the algorithm</b>; it is not prescribed in advance.

Several independent random starting vectors are tried. If none of the
resulting filters satisfies all specifications, the solution with the smallest
maximum weighted error is retained.

Finally, unstable poles are reflected inside the unit circle, the normalization
constant is corrected automatically, and the final eighth-order transfer
function is constructed symbolically with SymPy.

<div style="margin-top:8px;padding:8px 10px;background:#fff9e8;border:1px solid #d9c477;border-radius:6px;">

<b>Implementation note.</b>
This notebook is an independent Python implementation of the
optimization-based IIR design procedure. The numerical results are not
expected to reproduce the MATLAB example coefficient by coefficient. The
original implementation uses MATLAB <code>fminunc</code> with its own
optimization strategy and a randomly initialized parameter vector, whereas
this notebook uses SciPy's BFGS quasi-Newton optimizer, a reproducible random
initialization, and a least-p sequence with progressively increasing values of
p. Consequently, the number of iterations, the optimized parameter vector, and
the final filter coefficients may differ, although both implementations solve
the same design problem and are evaluated against the same frequency-domain
specifications.

<br><br>

The purpose of this notebook is therefore not to reproduce a particular
MATLAB run, but to demonstrate computationally the complete optimization
process from the design specifications to the final stable IIR filter.

</div>

</div>

</div>
"""))

# ============================================================
# DESIGN SPECIFICATIONS
# ============================================================

fp = 500.0
fsb = 800.0
Fs = 2000.0
Ap_required = 0.25
As_required = 45.0
N = 8

# ============================================================
# OPTIMIZATION SETTINGS
# ============================================================

eps1 = 1e-7
eps2 = 1e-7

p_initial = 2.0
mu = 2.0

number_of_attempts = 3

max_outer_iterations = 25
max_inner_iterations = 500

base_random_seed = 12345

# ============================================================
# PROBLEM DIMENSIONS
# ============================================================

number_of_sections = N//2

number_of_parameters = 4*number_of_sections+1

number_of_frequency_samples = 20*number_of_parameters

nyquist = Fs/2.0

# ============================================================
# FREQUENCY GRID
# ============================================================

active_bandwidth = fp+(nyquist-fsb)

frequency_step = active_bandwidth/(number_of_frequency_samples-2)

number_passband = int(np.floor(fp/frequency_step+0.5))+1

number_passband = max(2,min(number_passband,number_of_frequency_samples-2))

number_stopband = number_of_frequency_samples-number_passband

frequency_passband = np.linspace(0.0,fp,number_passband)

frequency_stopband = np.linspace(fsb,nyquist,number_stopband)

frequency_grid = np.concatenate((frequency_passband,frequency_stopband))

omega_grid = 2.0*np.pi*frequency_grid/Fs

desired_magnitude = np.concatenate((np.ones(number_passband),np.zeros(number_stopband)))

# ============================================================
# WEIGHTING FACTORS
# ============================================================

epsilon_p = (10.0**(0.05*Ap_required)-1.0)/(10.0**(0.05*Ap_required)+1.0)

epsilon_s = 10.0**(-0.05*As_required)

passband_weight = 1.0

stopband_weight = epsilon_p/epsilon_s

weights = np.concatenate((np.full(number_passband,passband_weight),np.full(number_stopband,stopband_weight)))

# ============================================================
# FREQUENCY RESPONSE OF THE CASCADED SECOND-ORDER MODEL
# ============================================================

def model_magnitude(x,omega):

    z = np.exp(1j*omega)

    z2 = z*z

    magnitude = np.full(omega.shape,abs(x[-1]),dtype=float)

    for section in range(number_of_sections):

        alpha0 = x[4*section]
        alpha1 = x[4*section+1]
        beta0 = x[4*section+2]
        beta1 = x[4*section+3]

        numerator = alpha0+alpha1*z+z2

        denominator = beta0+beta1*z+z2

        denominator_magnitude = np.abs(denominator)

        if np.any(denominator_magnitude < 1e-12):

            return np.full(omega.shape,1e12,dtype=float)

        magnitude *= np.abs(numerator)/denominator_magnitude

    return magnitude

# ============================================================
# WEIGHTED ERROR VECTOR
# ============================================================

def weighted_error(x):

    magnitude = model_magnitude(x,omega_grid)

    if not np.all(np.isfinite(magnitude)):

        return np.full_like(omega_grid,1e12,dtype=float)

    return weights*(magnitude-desired_magnitude)

# ============================================================
# STABLE NUMERICAL FORM OF THE LEAST-p OBJECTIVE
# ============================================================

def least_p_objective(x,p):

    if np.any(~np.isfinite(x)):

        return 1e12

    if np.max(np.abs(x)) > 100.0:

        return 1e12

    absolute_error = np.abs(weighted_error(x))

    maximum_error = np.max(absolute_error)

    if not np.isfinite(maximum_error):

        return 1e12

    if maximum_error < 1e-15:

        return 0.0

    normalized_error = absolute_error/maximum_error

    return maximum_error*np.sum(normalized_error**p)**(1.0/p)

# ============================================================
# GENERATE RANDOM INITIAL VECTOR
# ============================================================

def generate_initial_vector(seed):

    rng = np.random.default_rng(seed)

    x0 = np.empty(number_of_parameters)

    x0[:-1] = rng.uniform(0.05,1.50,number_of_parameters-1)

    x0[-1] = rng.uniform(0.10,1.00)

    return x0

# ============================================================
# RUN ONE COMPLETE LEAST-p OPTIMIZATION ATTEMPT
# ============================================================

def run_optimization_attempt(seed):

    x = generate_initial_vector(seed)

    p = p_initial

    previous_maximum_error = np.inf

    outer_history = []

    all_inner_values = []

    converged = False

    for outer_iteration in range(1,max_outer_iterations+1):

        inner_values = []

        def objective_current(x_current):

            return least_p_objective(x_current,p)

        def callback(x_current):

            value = objective_current(x_current)

            inner_values.append(value)

            all_inner_values.append(value)

        result = minimize(objective_current,x,method='BFGS',callback=callback,options={'gtol':eps2,'maxiter':max_inner_iterations,'disp':False})

        x_new = result.x.copy()

        x_new[-1] = abs(x_new[-1])

        current_error_vector = weighted_error(x_new)

        current_maximum_error = np.max(np.abs(current_error_vector))

        delta_x = np.linalg.norm(x_new-x)

        outer_history.append({
            'iteration':outer_iteration,
            'p':p,
            'objective':least_p_objective(x_new,p),
            'maximum_error':current_maximum_error,
            'delta_x':delta_x,
            'inner_iterations':result.nit,
            'optimizer_success':result.success
        })

        if abs(previous_maximum_error-current_maximum_error) < eps1:

            x = x_new

            converged = True

            break

        x = x_new

        previous_maximum_error = current_maximum_error

        p *= mu

    return {
        'seed':seed,
        'x0':generate_initial_vector(seed),
        'x':x,
        'history':outer_history,
        'inner_values':all_inner_values,
        'converged':converged,
        'maximum_weighted_error':np.max(np.abs(weighted_error(x)))
    }

# ============================================================
# SECOND-ORDER SECTIONS FROM THE OPTIMIZED VECTOR
# ============================================================

def sections_from_vector(x):

    sections = []

    for section in range(number_of_sections):

        alpha0 = x[4*section]
        alpha1 = x[4*section+1]
        beta0 = x[4*section+2]
        beta1 = x[4*section+3]

        numerator = np.array([1.0,alpha1,alpha0],dtype=float)

        denominator = np.array([1.0,beta1,beta0],dtype=float)

        sections.append((numerator,denominator))

    return sections

# ============================================================
# BUILD COMPLETE TRANSFER FUNCTION FROM SECTIONS
# ============================================================

def combine_sections(sections,H0):

    numerator = np.array([1.0])

    denominator = np.array([1.0])

    for b_section,a_section in sections:

        numerator = np.convolve(numerator,b_section)

        denominator = np.convolve(denominator,a_section)

    numerator = H0*numerator

    return numerator,denominator

# ============================================================
# STABILITY CORRECTION
# ============================================================

def stabilize_solution(x):

    original_sections = sections_from_vector(x)

    stable_sections = []

    original_poles = []

    stable_poles = []

    section_data = []

    H0_stable = abs(x[-1])

    for section_number,(numerator,denominator) in enumerate(original_sections,1):

        poles = np.roots(denominator)

        corrected_poles = []

        number_corrected = 0

        for pole in poles:

            original_poles.append(pole)

            if abs(pole) > 1.0:

                H0_stable /= abs(pole)

                corrected_pole = 1.0/np.conj(pole)

                number_corrected += 1

            else:

                corrected_pole = pole

            corrected_poles.append(corrected_pole)

            stable_poles.append(corrected_pole)

        corrected_denominator = np.real_if_close(np.poly(corrected_poles),tol=1000).astype(float)

        stable_sections.append((numerator.copy(),corrected_denominator))

        section_data.append({
            'section':section_number,
            'numerator':numerator.copy(),
            'original_denominator':denominator.copy(),
            'stable_denominator':corrected_denominator.copy(),
            'original_poles':np.array(poles),
            'stable_poles':np.array(corrected_poles),
            'number_corrected':number_corrected
        })

    b_original,a_original = combine_sections(original_sections,abs(x[-1]))

    b_stable,a_stable = combine_sections(stable_sections,H0_stable)

    return {
        'original_sections':original_sections,
        'stable_sections':stable_sections,
        'section_data':section_data,
        'H0_original':abs(x[-1]),
        'H0_stable':H0_stable,
        'b_original':b_original,
        'a_original':a_original,
        'b_stable':b_stable,
        'a_stable':a_stable,
        'original_poles':np.array(original_poles),
        'stable_poles':np.array(stable_poles)
    }

# ============================================================
# MEASURE FINAL FILTER SPECIFICATIONS
# ============================================================

def measure_filter(b,a):

    frequency,H = freqz(b,a,worN=65536,fs=Fs)

    magnitude = np.abs(H)

    magnitude_db = 20.0*np.log10(np.maximum(magnitude,1e-14))

    passband_mask = frequency <= fp

    stopband_mask = frequency >= fsb

    maximum_passband_deviation = np.max(np.abs(magnitude_db[passband_mask]))

    passband_peak_to_peak = np.max(magnitude_db[passband_mask])-np.min(magnitude_db[passband_mask])

    stopband_attenuation = -np.max(magnitude_db[stopband_mask])

    return {
        'frequency':frequency,
        'magnitude':magnitude,
        'magnitude_db':magnitude_db,
        'maximum_passband_deviation':maximum_passband_deviation,
        'passband_peak_to_peak':passband_peak_to_peak,
        'stopband_attenuation':stopband_attenuation,
        'pass':maximum_passband_deviation <= Ap_required and stopband_attenuation >= As_required
    }

# ============================================================
# RUN ALL RANDOM ATTEMPTS
# ============================================================

attempts = []

for attempt_index in range(number_of_attempts):

    seed = base_random_seed+attempt_index

    attempt = run_optimization_attempt(seed)

    stabilized = stabilize_solution(attempt['x'])

    measurements = measure_filter(stabilized['b_stable'],stabilized['a_stable'])

    attempt['stabilized'] = stabilized

    attempt['measurements'] = measurements

    attempts.append(attempt)

# ============================================================
# SELECT THE BEST ATTEMPT
# ============================================================

successful_attempts = [attempt for attempt in attempts if attempt['measurements']['pass']]

if successful_attempts:

    best_attempt = min(successful_attempts,key=lambda item:item['maximum_weighted_error'])

else:

    best_attempt = min(attempts,key=lambda item:item['maximum_weighted_error'])

x_final = best_attempt['x']

stabilized = best_attempt['stabilized']

measurements = best_attempt['measurements']

# ============================================================
# SYMBOLIC CONSTRUCTION OF THE FINAL TRANSFER FUNCTION
# ============================================================

q = sp.symbols('q')

symbolic_numerator = sp.Float(stabilized['H0_stable'],16)

symbolic_denominator = sp.Integer(1)

for numerator,denominator in stabilized['stable_sections']:

    numerator_symbolic = sp.Float(numerator[0],16)+sp.Float(numerator[1],16)*q+sp.Float(numerator[2],16)*q**2

    denominator_symbolic = sp.Float(denominator[0],16)+sp.Float(denominator[1],16)*q+sp.Float(denominator[2],16)*q**2

    symbolic_numerator = sp.expand(symbolic_numerator*numerator_symbolic)

    symbolic_denominator = sp.expand(symbolic_denominator*denominator_symbolic)

symbolic_numerator = sp.expand(symbolic_numerator)

symbolic_denominator = sp.expand(symbolic_denominator)

b_symbolic = sp.Poly(symbolic_numerator,q).all_coeffs()[::-1]

a_symbolic = sp.Poly(symbolic_denominator,q).all_coeffs()[::-1]

b_final = np.array([float(value) for value in b_symbolic])

a_final = np.array([float(value) for value in a_symbolic])

# ============================================================
# DISPLAY — BASIC OPTIMIZATION INFORMATION
# ============================================================

attempt_rows = ""

for index,attempt in enumerate(attempts,1):

    m = attempt['measurements']

    attempt_rows += f"""
    <tr>
        <td>{index}</td>
        <td>{attempt['seed']}</td>
        <td>{len(attempt['history'])}</td>
        <td>{attempt['maximum_weighted_error']:.3e}</td>
        <td>{m['maximum_passband_deviation']:.4f} dB</td>
        <td>{m['stopband_attenuation']:.2f} dB</td>
        <td><b>{"PASS" if m['pass'] else "FAIL"}</b></td>
    </tr>
    """

best_index = attempts.index(best_attempt)+1

display(HTML(f"""
<div class="io-root">

<div class="io-box io-note">

<div class="io-title">Optimization attempts</div>

<table class="io-table">
<tr>
    <th>Attempt</th>
    <th>Seed</th>
    <th>Outer iterations</th>
    <th>Max weighted error</th>
    <th>Passband deviation</th>
    <th>Stopband attenuation</th>
    <th>Specs</th>
</tr>

{attempt_rows}

</table>

<div style="margin-top:7px;">
Selected attempt: <b>{best_index}</b>
&nbsp;&nbsp;
Frequency samples: <b>{number_of_frequency_samples}</b>
&nbsp;&nbsp;
Passband samples: <b>{number_passband}</b>
&nbsp;&nbsp;
Stopband samples: <b>{number_stopband}</b>
</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — CALCULATED PARAMETER VECTOR
# ============================================================

xi_string = ', '.join([f'{value:.6f}' for value in x_final])

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">Calculated optimization vector ξ</div>

<div class="io-code" style="line-height:1.55;">
[{xi_string}]
</div>

<div class="io-small" style="margin-top:6px;">
This vector was calculated by the Python optimization procedure; it was not
entered as input data.
</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — OPTIMIZATION HISTORY
# ============================================================

history_rows = ""

for item in best_attempt['history']:

    history_rows += f"""
    <tr>
        <td>{item['iteration']}</td>
        <td>{item['p']:.0f}</td>
        <td>{item['inner_iterations']}</td>
        <td>{item['objective']:.3e}</td>
        <td>{item['maximum_error']:.3e}</td>
        <td>{item['delta_x']:.3e}</td>
    </tr>
    """

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">Least-p iteration history of the selected attempt</div>

<table class="io-table">
<tr>
    <th>k</th>
    <th>p</th>
    <th>BFGS iterations</th>
    <th>Ψ<sub>p</sub></th>
    <th>max |e|</th>
    <th>||δx||₂</th>
</tr>

{history_rows}

</table>

</div>

</div>
"""))

# ============================================================
# DISPLAY — AUTOMATIC STABILITY CORRECTION
# ============================================================

section_rows = ""

for data in stabilized['section_data']:

    original_radius = np.max(np.abs(data['original_poles']))

    stable_radius = np.max(np.abs(data['stable_poles']))

    section_rows += f"""
    <tr>
        <td>H{data['section']}</td>
        <td>{original_radius:.6f}</td>
        <td>{data['number_corrected']}</td>
        <td>{stable_radius:.6f}</td>
    </tr>
    """

display(HTML(f"""
<div class="io-root">

<div class="io-box io-note">

<div class="io-title">Automatic pole-stability correction</div>

<table class="io-table">
<tr>
    <th>Section</th>
    <th>Original max |p|</th>
    <th>Reflected poles</th>
    <th>Final max |p|</th>
</tr>

{section_rows}

</table>

<div style="margin-top:7px;">
Calculated H₀ before correction:
<b>{stabilized['H0_original']:.6f}</b>
&nbsp;&nbsp;&nbsp;
Calculated H₀ after correction:
<b>{stabilized['H0_stable']:.6f}</b>
</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — CALCULATED STABILIZED SECOND-ORDER SECTIONS
# ============================================================

sections_html = ""

for index,(numerator,denominator) in enumerate(stabilized['stable_sections'],1):

    sections_html += f"""
    H<sub>{index}</sub>(z) =
    (1 {numerator[1]:+.6f}z<sup>-1</sup> {numerator[2]:+.6f}z<sup>-2</sup>) /
    (1 {denominator[1]:+.6f}z<sup>-1</sup> {denominator[2]:+.6f}z<sup>-2</sup>)
    <br>
    """

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">Calculated stabilized second-order sections</div>

<div class="io-code" style="line-height:1.8;">
{sections_html}
</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — FINAL SYMBOLIC COEFFICIENTS
# ============================================================

b_string = ', '.join([f'{value:.6f}' for value in b_final])

a_string = ', '.join([f'{value:.6f}' for value in a_final])

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">Final eighth-order transfer function</div>

The final numerator and denominator are obtained by symbolic multiplication
of the calculated second-order sections.

<br><br>

<b>Numerator coefficients b:</b>

<div class="io-code">
[{b_string}]
</div>

<br>

<b>Denominator coefficients a:</b>

<div class="io-code">
[{a_string}]
</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — FINAL SPECIFICATION CHECK
# ============================================================

passband_pass = measurements['maximum_passband_deviation'] <= Ap_required

stopband_pass = measurements['stopband_attenuation'] >= As_required

display(HTML(f"""
<div class="io-root">

<div class="io-box io-note">

<div class="io-title">Final specification check</div>

<div class="io-cols">

<div class="io-col">

<b>Maximum passband deviation</b><br>
{measurements['maximum_passband_deviation']:.6f} dB<br>

Required:
≤ {Ap_required:.2f} dB<br>

Result:
<b>{"PASS" if passband_pass else "FAIL"}</b>

</div>

<div class="io-col">

<b>Passband peak-to-peak ripple</b><br>
{measurements['passband_peak_to_peak']:.6f} dB

</div>

<div class="io-col">

<b>Stopband attenuation</b><br>
{measurements['stopband_attenuation']:.6f} dB<br>

Required:
≥ {As_required:.0f} dB<br>

Result:
<b>{"PASS" if stopband_pass else "FAIL"}</b>

</div>

</div>

</div>

</div>
"""))

# ============================================================
# DATA FOR PLOTS
# ============================================================

best_history = best_attempt['history']

iteration_axis = np.array([item['iteration'] for item in best_history])

maximum_error_history = np.array([item['maximum_error'] for item in best_history])

delta_history = np.array([item['delta_x'] for item in best_history])

frequency = measurements['frequency']

magnitude_db = measurements['magnitude_db']

theta = np.linspace(0,2*np.pi,1000)

original_poles = stabilized['original_poles']

stable_poles = stabilized['stable_poles']

# ============================================================
# FIGURE — 2 x 2 GRID
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.4))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. MAXIMUM ERROR VS OUTER ITERATION
# ============================================================

ax1.plot(iteration_axis,maximum_error_history,'o-',color='red',linewidth=1.3,markersize=4)

ax1.set_yscale('log')

ax1.set_xlim(0.5,max(iteration_axis)+0.5)

ax1.set_title('Maximum Weighted Error vs Outer Iteration')

ax1.set_xlabel('Outer iteration k')

ax1.set_ylabel(r'$\max |e_i|$')

ax1.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# 2. PARAMETER-VECTOR CORRECTION
# ============================================================

ax2.plot(iteration_axis,delta_history,'o-',color='red',linewidth=1.3,markersize=4)

ax2.set_yscale('log')

ax2.set_xlim(0.5,max(iteration_axis)+0.5)

ax2.set_title(r'Parameter Correction $\|\delta x_k\|_2$')

ax2.set_xlabel('Outer iteration k')

ax2.set_ylabel(r'$\|\delta x_k\|_2$')

ax2.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# 3. POLES BEFORE AND AFTER STABILIZATION
# ============================================================

ax3.plot(np.cos(theta),np.sin(theta),'--',linewidth=1.0,label='Unit circle')

ax3.plot(np.real(original_poles),np.imag(original_poles),'x',markersize=7,markeredgewidth=1.5,label='Before correction')

ax3.plot(np.real(stable_poles),np.imag(stable_poles),'ro',markersize=4,fillstyle='none',label='After correction')

ax3.axhline(0,color='black',linewidth=0.8)

ax3.axvline(0,color='black',linewidth=0.8)

pole_limit = max(1.2,1.1*np.max(np.abs(original_poles)))

ax3.set_xlim(-pole_limit,pole_limit)

ax3.set_ylim(-pole_limit,pole_limit)

ax3.set_aspect('equal',adjustable='box')

ax3.set_title('Pole Stability Correction')

ax3.set_xlabel(r'$\Re\{z\}$')

ax3.set_ylabel(r'$\Im\{z\}$')

ax3.grid(True,linestyle=':',alpha=0.25)

ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 4. FINAL FREQUENCY RESPONSE
# ============================================================

ax4.plot(frequency,magnitude_db,color='red',linewidth=1.3,label='Calculated IIR filter')

ax4.axvline(fp,linestyle='--',linewidth=1.0,label=r'$f_p$')

ax4.axvline(fsb,linestyle=':',linewidth=1.0,label=r'$f_s$')

ax4.axhline(-Ap_required,linestyle='--',linewidth=0.9,label=r'$-A_p$')

ax4.axhline(-As_required,linestyle=':',linewidth=0.9,label=r'$-A_s$')

ax4.set_xlim(0,Fs/2)

ax4.set_ylim(-100,5)

ax4.set_title('Final Optimized IIR Magnitude Response')

ax4.set_xlabel('Frequency (Hz)')

ax4.set_ylabel('Magnitude (dB)')

ax4.grid(True,linestyle=':',alpha=0.25)

ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# LAYOUT
# ============================================================

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.28,hspace=0.56)

# ============================================================
# DISPLAY
# ============================================================

display(fig.canvas)